# Evaluate the mouse MERFISH MOp cortex benchmark

Gather per-seed predictions from `results/<NAME>/<Method>/` and compute four
metrics against the Target ground truth:

- **accuracy**
- **weighted F1**
- **ARI** (adjusted Rand index)
- **macro F1**

The metric logic mirrors `Benchmarks_new/plot_scripts/utils.py::gather_metrics`
so numbers are comparable to the main study:

| method     | prediction source (per seed) |
|------------|------------------------------|
| NovAST     | `seed{i}/adata_unlabeled_final.h5ad` → `obs['voted_final_prediction']` vs `obs['ground_truth']` |
| STELLAR    | `seed{i}/prediction.npy` vs `seed{i}/ground_truth.npy` |
| scBOL      | `seed{i}/prediction.npy` vs `seed{i}/ground_truth.npy` |
| SpatialID  | `seed{i}/annotation.h5ad` → `obs['celltype_pred']` vs `obs[celltype_name_target]` |
| Tangram    | per-slice `seed{i}_gt_te_*.npy` / `seed{i}_pred_te_*.npy` (averaged over slices) |

Integer cluster labels are remapped to names via `inverse_dict_reference.pkl`, then
Hungarian-matched to unseen novel clusters via `calculate_optimal_accuracy_final`.


In [ ]:
import os, glob, re, pickle
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix, f1_score, adjusted_rand_score

# BASE is this benchmark directory (the notebook's own folder). Set it manually
# if you run this notebook from a different working directory.
BASE     = os.getcwd()
assert os.path.isdir(os.path.join(BASE, "results")), (
    f"Expected to run from the benchmark/ directory, but cwd is {BASE}. "
    "Set BASE to the benchmark directory explicitly."
)
RESULTS  = os.path.join(BASE, "results")

SETTING  = "most"
NAME     = f"merfish_split_{SETTING}"
ROOT     = os.path.join(RESULTS, NAME)
CELLTYPE_NAME_TARGET = "cell_type"
MAX_SEED = 10

with open(os.path.join(ROOT, "inverse_dict_reference.pkl"), "rb") as f:
    INVERSE_DICT = pickle.load(f)
print("run dir:", ROOT)
print("reference classes:", len(INVERSE_DICT))


In [ ]:
# --- metric helpers (copied from plot_scripts/utils.py) ---

def calculate_optimal_accuracy_final(true_labels, cluster_labels, inverse_dict):
    clusters_to_map    = {c for c in set(cluster_labels) if c not in set(inverse_dict.values())}
    true_labels_to_map = set(true_labels) - set(cluster_labels)
    all_labels = list(true_labels_to_map.union(clusters_to_map))
    confusion  = confusion_matrix(true_labels, cluster_labels, labels=all_labels)
    if len(clusters_to_map) == len(true_labels_to_map):
        row_ind, col_ind = linear_sum_assignment(confusion, maximize=True)
        mapping = {all_labels[c]: all_labels[r] for r, c in zip(row_ind, col_ind)}
    else:
        mapping, assigned = {}, set()
        for tl in true_labels_to_map:
            ti = all_labels.index(tl)
            overlaps = confusion[ti, :]
            for cl in clusters_to_map:
                ci = all_labels.index(cl)
                if cl not in assigned and overlaps[ci] == overlaps.max():
                    mapping[cl] = tl; assigned.add(cl); break
    return np.array([mapping.get(l, l) for l in cluster_labels]), mapping

def four_metrics(gts, preds):
    return dict(
        accuracy    = float((gts == preds).mean()),
        weighted_F1 = float(f1_score(gts, preds, average="weighted", zero_division=0)),
        ARI         = float(adjusted_rand_score(gts, preds)),
        macro_F1    = float(f1_score(gts, preds, average="macro", zero_division=0)),
    )

# --- Tangram: mirror plot_scripts/utils.py exactly ------------------------
# The benchmark averages in TWO levels:
#   1. compute_tangram_slice_metrics -> metrics per file, averaged WITHIN each
#      test slice `te`  (matters only in the retired tr x te cross-product mode,
#      where several reference slices map onto the same target slice)
#   2. preprocess_df -> unweighted mean ACROSS te -> one value per seed
# For merfish_split there is exactly 1 file per te (17 files / 17 tags), so both
# levels collapse to a plain mean over slices -- but we replicate the grouping so
# this stays faithful for whole/chunked/cross-product layouts too.
#
# NOTE: this is an UNWEIGHTED mean over slices, so a 500-cell slice counts as much
# as a 10,000-cell one. That is the benchmark's definition, not overall accuracy;
# we match it so Tangram stays comparable to the published numbers.
def tangram_seed_metrics(dir_path, seed, inverse_dict):
    per_slice = {}                       # te -> metric -> [values]
    for gt_path in glob.glob(os.path.join(dir_path, f"seed{seed}_gt_*.npy")):
        base = os.path.basename(gt_path)
        if "_te_" in base and "_tr_" in base:          # cross-product (retired)
            te = re.search(r"_te_(.+)\.npy$", base).group(1)
            pred_path = gt_path.replace("_gt_tr_", "_pred_tr_")
        elif "_te_" in base:                            # region-by-region (merfish)
            te = re.search(r"_te_(.+)\.npy$", base).group(1)
            pred_path = gt_path.replace("_gt_te_", "_pred_te_")
        elif base.endswith("_gt_whole.npy"):            # whole-slide
            te = "whole"
            pred_path = gt_path.replace("_gt_whole.npy", "_pred_whole.npy")
        else:                                           # spatial-KMeans chunks
            m = re.search(rf"seed{seed}_gt_(.+)\.npy$", base)
            te = m.group(1) if m else base
            pred_path = gt_path.replace("_gt_", "_pred_")
        if not os.path.exists(pred_path):
            continue
        gt   = np.load(gt_path,  allow_pickle=True).astype(str)
        pred = np.load(pred_path, allow_pickle=True)
        pred = np.array([inverse_dict.get(x, x) for x in pred], dtype=str)
        mets = per_slice.setdefault(te, {})
        for k, v in four_metrics(gt, pred).items():
            mets.setdefault(k, []).append(v)
    if not per_slice:
        return None
    per_te = {te: {k: float(np.mean(v)) for k, v in m.items()}
              for te, m in per_slice.items()}                      # level 1
    keys = next(iter(per_te.values())).keys()
    return {k: float(np.mean([m[k] for m in per_te.values()]))      # level 2
            for k in keys}


In [ ]:
def load_gts_preds(method, seed):
    d = os.path.join(ROOT, method)
    if method == "STELLAR" or method.startswith("scBOL"):
        preds = np.load(os.path.join(d, f"seed{seed}/prediction.npy"),   allow_pickle=True)
        gts   = np.load(os.path.join(d, f"seed{seed}/ground_truth.npy"), allow_pickle=True)
    elif method == "NovAST":
        ad    = sc.read_h5ad(os.path.join(d, f"seed{seed}/adata_unlabeled_final.h5ad"))
        preds = ad.obs["voted_final_prediction"].to_numpy()
        gts   = ad.obs["ground_truth"].to_numpy()
    elif method == "SpatialID":
        ad    = sc.read_h5ad(os.path.join(d, f"seed{seed}/annotation.h5ad"))
        gts   = ad.obs[CELLTYPE_NAME_TARGET].to_numpy()
        preds = ad.obs["celltype_pred"].to_numpy()
    elif method == "SingleR":
        # SingleR is deterministic and has no seed subfolders: one result at the
        # method root (written by SingleR/singler_io.py convert).
        preds = np.load(os.path.join(d, "prediction.npy"),   allow_pickle=True)
        gts   = np.load(os.path.join(d, "ground_truth.npy"), allow_pickle=True)
    else:
        raise ValueError(method)
    return np.asarray(gts).astype(str), np.asarray(preds)

def seed_metrics(method, seed):
    if method == "Tangram":
        return tangram_seed_metrics(os.path.join(ROOT, method) + "/", seed, INVERSE_DICT)
    gts, preds = load_gts_preds(method, seed)
    preds = np.array([INVERSE_DICT.get(x, x) for x in preds], dtype=str)
    try:
        preds, _ = calculate_optimal_accuracy_final(gts, preds, INVERSE_DICT)
    except Exception:
        pass
    return four_metrics(gts, preds)

METHODS = ["NovAST", "STELLAR", "scBOL", "SpatialID", "Tangram", "SingleR"]
rows = []
for method in METHODS:
    if not os.path.isdir(os.path.join(ROOT, method)):
        continue
    for seed in range(1, MAX_SEED + 1):
        if method == "SingleR" and seed > 1:
            continue      # no seeds -- scored once
        try:
            m = seed_metrics(method, seed)
        except Exception:
            m = None
        if m:
            rows.append({"method": method, "seed": seed, **m})

df = pd.DataFrame(rows)
print(f"gathered {len(df)} (method, seed) results")
df.head(20)


In [ ]:
# Summary: mean +/- std across seeds, per method
if len(df):
    metric_cols = ["accuracy", "weighted_F1", "ARI", "macro_F1"]
    summary = (df.groupby("method")[metric_cols]
                 .agg(["mean", "std", "count"])
                 .round(3))
    display(summary)
else:
    print("No results yet — run notebook 01 first.")


In [ ]:
# Canonical palette + order, matching Benchmarks_new/plot_scripts/utils.py
import matplotlib.pyplot as plt

FULL_PALETTE = {
    'NovAST':     '#619b8a',
    'Tangram':    '#ffe97f',
    'Spatial-ID': '#ff9770',
    'scBOL':      '#957fef',
    'SingleR':    '#81c3d7',
    'STELLAR':    '#e0aaff',
}
METHOD_ORDER = ['NovAST', 'SingleR', 'Tangram', 'Spatial-ID', 'STELLAR', 'scBOL']
# results-dir name -> canonical display label (as preprocess_df does)
DISPLAY_NAME = {'SpatialID': 'Spatial-ID'}
METRIC_LABEL = {'accuracy': 'Accuracy', 'weighted_F1': 'Weighted F1',
                'ARI': 'ARI', 'macro_F1': 'Macro F1'}

if len(df):
    metric_cols = ["accuracy", "weighted_F1", "ARI", "macro_F1"]
    dfp = df.copy()
    dfp["method"] = dfp["method"].replace(DISPLAY_NAME)

    order = [m for m in METHOD_ORDER if m in set(dfp["method"])]
    means = dfp.groupby("method")[metric_cols].mean().reindex(order)
    stds  = dfp.groupby("method")[metric_cols].std().reindex(order)
    colors = [FULL_PALETTE[m] for m in order]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    for ax, col in zip(axes, metric_cols):
        ax.bar(order, means[col].values, yerr=stds[col].values,
               color=colors, edgecolor="black", linewidth=0.5, capsize=4)
        ax.set_title(METRIC_LABEL[col])
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=45, ha="right")
        ax.spines[["top", "right"]].set_visible(False)

    # y-axis 0-1 by default; only drop the floor if a metric actually goes
    # negative (ARI can) so the bars stay comparable across panels.
    ymin = min(0, float(np.nanmin(means.values - np.nan_to_num(stds.values))))
    axes[0].set_ylim(-0.2 if ymin < 0 else 0, 1)
    axes[0].set_ylabel("score")
    fig.suptitle(f"MERFISH split — setting = {SETTING}  (bars: mean over seeds, error bars: SD)")
    fig.tight_layout()
    plt.show()
else:
    print("No results to plot yet.")


To compare **settings** (`none` / `most` / `least` / `closest` / `furthest`),
run notebook 01 for each, then loop this gather over the `NAME`s and concatenate
the resulting DataFrames with a `setting` column.
